# ForenSight - AI-Image Detector (GenImage benchmark)

Trains the detector on **Tiny-GenImage** - a Colab-sized slice of the
**GenImage** benchmark (NeurIPS 2023). 35,000 images across **8 generators**
(ADM, BigGAN, GLIDE, Midjourney, SD1.4, SD1.5, VQDM, Wukong) plus real photos.

**This version saves the model to your Google Drive after every epoch**, so if
Colab's free tier disconnects, you just reconnect and grab the file - you never
lose your training.

### How to run
1. Runtime -> Change runtime type -> **GPU (T4)**.
2. Run **cell 1** first - it installs `datasets` and **restarts the runtime
   once on purpose** (*'session restarted'* is normal).
3. After it restarts, run every cell from **cell 2 downward**, in order.
   Do **not** re-run cell 1.
4. Keep this browser tab open and your laptop awake while it trains, or Colab
   disconnects for inactivity.


## 1. Install datasets + restart the runtime (run this ONCE, first)
Colab ships an older `pyarrow`; `datasets` needs a newer one, and swapping it
needs a kernel restart. **This cell restarts on purpose** - you will see
*'Your session crashed / restarted'*. That is expected. When it comes back,
**skip this cell** and continue from cell 2.


In [ ]:
!pip install -q -U 'datasets' 'pyarrow>=14'
print('Installed. Restarting the runtime once (this is normal)...')
print('When it restarts, DO NOT re-run this cell - continue from cell 2.')
import os
os.kill(os.getpid(), 9)   # force a clean restart so the new pyarrow loads

## 2. Check the GPU is on


In [ ]:
import tensorflow as tf
print('TensorFlow:', tf.__version__)
print('GPU available:', tf.config.list_physical_devices('GPU'))

## 3. Mount Google Drive (so your model survives disconnects)
A popup will ask you to authorise access to your own Drive - click through it.
The model gets saved to `My Drive/ForenSight/`. If Colab later disconnects,
the newest model is already sitting safely in your Drive.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
SAVE_DIR = '/content/drive/MyDrive/ForenSight'
os.makedirs(SAVE_DIR, exist_ok=True)
MODEL_PATH = os.path.join(SAVE_DIR, 'ai_detector_genimage.keras')
print('Model will be saved to:', MODEL_PATH)

## 4. Download Tiny-GenImage (Hugging Face - no key needed)
Public dataset. Loads into memory-mapped Arrow (does **not** all load to RAM).


In [ ]:
from datasets import load_dataset
ds = load_dataset('TheKernel01/Tiny-GenImage')
print(ds)
train_hf, val_hf = ds['train'], ds['validation']
print('train:', len(train_hf), ' validation:', len(val_hf))
print('columns:', train_hf.column_names)

## 5. Understand the labels
Each row has an image, a **binary `label`** (0 = real, 1 = fake) and a
**`generator`** id (0 = real, 1..8 = the AI models).


In [ ]:
GEN_NAMES = ['Real', 'ADM', 'BigGAN', 'GLIDE', 'Midjourney',
             'SD14', 'SD15', 'VQDM', 'Wukong']
import collections
print('label counts (train):', collections.Counter(train_hf['label']))
print('generator counts (train):',
      {GEN_NAMES[g]: c for g, c in sorted(collections.Counter(train_hf['generator']).items())})

## 6. Peek at a few images


In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(11, 6))
for i in range(8):
    row = train_hf[i]
    plt.subplot(2, 4, i + 1)
    plt.imshow(row['image'])
    kind = 'REAL' if row['label'] == 0 else 'FAKE'
    plt.title(f"{kind} - {GEN_NAMES[row['generator']]}")
    plt.axis('off')
plt.tight_layout(); plt.show()

## 7. Streaming input pipeline (224x224)
Resizes on the fly with a generator instead of loading all 35k into RAM (this
avoids the OOM crash). Model outputs **P(REAL)** (target = `1 - label`) so
ForenSight's `ml_detector.py` works unchanged.


In [ ]:
import numpy as np
IMG_SIZE = (224, 224)
BATCH = 32
AUTOTUNE = tf.data.AUTOTUNE

def make_tf(hf_split, shuffle):
    n = len(hf_split)
    def gen():
        idx = np.arange(n)
        if shuffle:
            np.random.shuffle(idx)
        for i in idx:
            row = hf_split[int(i)]
            img = row['image'].convert('RGB').resize(IMG_SIZE)
            arr = np.asarray(img, dtype=np.float32)      # raw 0..255
            target = 1.0 - float(row['label'])           # real=1, fake=0 -> P(REAL)
            yield arr, target
    sig = (tf.TensorSpec(shape=IMG_SIZE + (3,), dtype=tf.float32),
           tf.TensorSpec(shape=(), dtype=tf.float32))
    d = tf.data.Dataset.from_generator(gen, output_signature=sig)
    return d.batch(BATCH).prefetch(AUTOTUNE)

train_ds = make_tf(train_hf, shuffle=True)
val_ds = make_tf(val_hf, shuffle=False)
print('pipelines ready')

## 8. Build the model (transfer learning, MobileNetV2 @ 224)


In [ ]:
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import MobileNetV2

base = MobileNetV2(input_shape=IMG_SIZE + (3,), include_top=False, weights='imagenet')
base.trainable = False

inputs = layers.Input(shape=IMG_SIZE + (3,))
x = layers.RandomFlip('horizontal')(inputs)
x = layers.RandomRotation(0.05)(x)
x = layers.Rescaling(1.0 / 127.5, offset=-1)(x)      # 0..255 -> -1..1
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(1, activation='sigmoid')(x)   # P(REAL)
model = Model(inputs, outputs)
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

## 9. Train (frozen backbone) - saves to Drive every epoch
The `ModelCheckpoint` writes the **best** model to your Drive after each epoch.
So even if Colab disconnects mid-training, the best model so far is already
saved at `MODEL_PATH` - just reconnect and use it. This step alone gives you a
usable detector; step 10 (fine-tune) is an optional accuracy boost.


In [ ]:
ckpt = tf.keras.callbacks.ModelCheckpoint(
    MODEL_PATH, monitor='val_accuracy', save_best_only=True, verbose=1)
history = model.fit(train_ds, validation_data=val_ds, epochs=5, callbacks=[ckpt])
model.save(MODEL_PATH)   # also save the final model
print('Saved to', MODEL_PATH)

## 10. Fine-tune (optional accuracy boost) - also saves to Drive
Unfreezes the top layers for a few % more. Safe to skip if you are short on
time - the model from step 9 already works.


In [ ]:
base.trainable = True
for layer in base.layers[:-40]:
    layer.trainable = False
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
              loss='binary_crossentropy', metrics=['accuracy'])
history2 = model.fit(train_ds, validation_data=val_ds, epochs=5, callbacks=[ckpt])
model.save(MODEL_PATH)
print('Saved to', MODEL_PATH)

## 11. Per-generator accuracy (the key result for your report)
Shows how well the detector catches **each** generator - the table you cite
when you discuss the generalisation gap (and why your rule-based C2PA detector
is still needed for 2025 models like Grok / Gemini / GPT-image).


In [ ]:
probs, labels, gens = [], [], []
buf_imgs = []
BATCHP = 64
def flush(buf):
    if not buf:
        return []
    arr = np.stack(buf).astype(np.float32)
    return model.predict(arr, verbose=0).flatten().tolist()

for i in range(len(val_hf)):
    row = val_hf[i]
    img = row['image'].convert('RGB').resize(IMG_SIZE)
    buf_imgs.append(np.asarray(img, dtype=np.float32))
    labels.append(row['label']); gens.append(row['generator'])
    if len(buf_imgs) == BATCHP:
        probs.extend(flush(buf_imgs)); buf_imgs = []
probs.extend(flush(buf_imgs))

probs = np.array(probs); labels = np.array(labels); gens = np.array(gens)
pred_fake = (probs < 0.5).astype(int)   # low P(REAL) -> predicted fake

print(f'Overall accuracy: {(pred_fake == labels).mean():.3f}\n')
print(f'{"generator":<12}{"images":>8}{"detect%":>10}')
for g in range(len(GEN_NAMES)):
    m = gens == g
    if not m.any():
        continue
    acc = (pred_fake[m] == (0 if g == 0 else 1)).mean()
    print(f'{GEN_NAMES[g]:<12}{m.sum():>8}{acc*100:>9.1f}%')

## 12. Get the model onto your laptop
It is already in your Google Drive at `My Drive/ForenSight/`. Easiest: open
Google Drive on your laptop, download `ai_detector_genimage.keras`, rename to
**`ai_detector.keras`**, and drop it in `forensight/models/`. (The cell below
also triggers a direct browser download as a backup.)


In [ ]:
from google.colab import files
files.download(MODEL_PATH)

## 13. Try it on your own image
P(AI) = 1 - P(REAL).


In [ ]:
from google.colab import files
up = files.upload()
for fn in up:
    img = tf.keras.utils.load_img(fn, target_size=IMG_SIZE)
    arr = tf.keras.utils.img_to_array(img)[None, ...]
    p_real = float(model.predict(arr, verbose=0)[0][0])
    print(f'{fn}:  P(AI) = {(1 - p_real):.1%}   P(REAL) = {p_real:.1%}')

---
### If Colab disconnects mid-training
Your best model is already saved in Drive. To recover without retraining:
reconnect the runtime, then run cells **1 (skip), 2, 3, 4, 7, 8**, then load the
saved model instead of re-fitting:
```python
model = tf.keras.models.load_model(MODEL_PATH)
```
and jump straight to step 11 (evaluate) or step 12 (download).

### After you have the model
1. Rename `ai_detector_genimage.keras` -> **`ai_detector.keras`**, put it in
   `forensight/models/`.
2. `pip install tensorflow` locally, restart ForenSight -> the ML line uses it.
3. For your report: cite GenImage (NeurIPS 2023), include the per-generator
   table from step 11, and discuss the generalisation gap on 2025 models.
